# 금리 동결인데 올해 초보다 0.50%p 높다? 한국은행 기준금리 변경표로 확인하기

DataPopcorn 테크블로그 글의 재현 노트북입니다. `런타임 → 모두 실행`을 누르면 한국은행 기준금리 변경표에서 일별 금리를 복원하고, 글에 나온 숫자와 **차트 4개**를 다시 그립니다.

- 데이터: 한국은행 「한국은행 기준금리 추이」 변경일자 표(2026-10-23 조회)를 표 그대로 옮긴 CSV `bok_base_rate_table_raw.csv`. API 키가 필요 없습니다.
- 2026-10-22 동결 3.00%는 같은 날 한국은행 통화정책방향 결정문에서 확인한 값입니다. 금리가 바뀐 날이 아니므로 변경표에 행을 더하지 않습니다.
- 원문 HTML·PDF는 저작권 때문에 싣지 않았습니다. 주소와 SHA-256은 `source_metadata.json`에 있습니다.
- 단위: 기준금리는 연 %, 차이는 %p. 2026년은 10월 22일까지의 부분 연도입니다.
- 직접 다시 받아오는 코드는 맨 아래 부록에 있고 기본으로 꺼져 있습니다(`RUN_COLLECT = False`).

In [ ]:
# 0. 준비: 한글 폰트(Colab이면 나눔고딕 설치)와 데이터 위치
import os, sys, io, re, json, urllib.request
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.system('apt-get -qq install -y fonts-nanum > /dev/null')
for f in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf', '/System/Library/Fonts/AppleSDGothicNeo.ttc', 'C:/Windows/Fonts/malgun.ttf']:
    if os.path.exists(f):
        fm.fontManager.addfont(f)
        plt.rcParams['font.family'] = fm.FontProperties(fname=f).get_name()
        break
plt.rcParams.update({'axes.unicode_minus': False, 'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
C = {'main': '#E8590C', 'sub': '#1C7ED6', 'gray': '#868E96'}

# 공개 GitHub 저장소에서 읽습니다. 로컬 폴더를 쓰려면 DATA_DIR 환경변수를 지정하세요.
BASE = os.environ.get('DATA_DIR', 'https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/base-rate-hold-history/data')
def read_bytes(rel):
    p = f'{BASE}/{rel}'
    return urllib.request.urlopen(p).read() if p.startswith('http') else open(p, 'rb').read()
print('데이터 경로:', BASE)

## Step 1. 변경표를 그대로 읽습니다

한국은행 표는 **금리가 바뀐 날만** 한 줄씩 적습니다. 회의 목록이 아니어서, 동결한 회의는 표에 없습니다. 표는 최신순이고 연도·월일·금리가 글자로 들어 있습니다.

In [ ]:
raw = pd.read_csv(io.BytesIO(read_bytes('bok_base_rate_table_raw.csv')), dtype=str, encoding='utf-8-sig')
print('행 수:', len(raw), '/ 컬럼:', list(raw.columns))
print(raw.head(8).to_string(index=False))

## Step 2. 날짜·숫자로 바꾸고 오래된 순으로 정렬합니다

- 표 각주: 2008년 2월까지는 **콜금리 목표**, 2008년 3월부터는 **한국은행 기준금리**입니다. 정의가 다른 두 지표를 이어 붙이지 않으려고 경계 행의 차이는 계산하지 않습니다. 이 분석은 2024년부터라 앞쪽 행은 쓰지 않습니다.
- `diff_pp`는 직전 변경 대비 차이(%p)입니다.

In [ ]:
def parse_date(y, md):
    m = re.fullmatch(r'(\d{1,2})월 (\d{1,2})일', md.strip())
    return pd.Timestamp(int(y), int(m.group(1)), int(m.group(2)))

ch = raw.copy()
ch['date'] = [parse_date(y, md) for y, md in zip(ch.year_text, ch.month_day_text)]
ch['rate_pct'] = ch.base_rate_text.astype(float)
ch = ch.sort_values('date').reset_index(drop=True)
ch['series_type'] = np.where(ch.date < pd.Timestamp('2008-03-01'), '콜금리 목표', '한국은행 기준금리')
ch['diff_pp'] = ch.rate_pct.diff().round(2)
ch.loc[ch.series_type.ne(ch.series_type.shift()), 'diff_pp'] = np.nan
assert ch.date.is_unique
print(ch.series_type.value_counts().to_string())
print(ch[ch.date >= '2023-01-01'][['date', 'rate_pct', 'diff_pp']].to_string(index=False))

## Step 3. 변경표를 하루 단위 금리로 펼칩니다

각 날짜에는 **그날 이전(당일 포함) 가장 최근 변경값**을 붙입니다(forward-fill). 변경이 없던 날을 0으로 채우거나 두 변경 사이를 선형으로 잇지 않습니다. 금리는 계단처럼 움직이기 때문입니다.

2024-01-01의 값은 표의 2023-01-13 행(3.50%)에서 옵니다. 날짜는 표의 변경일자부터 새 금리를 적용한 계산이며, 법적 효력이 시작된 정확한 시각을 검증한 것은 아닙니다.

In [ ]:
START, END = pd.Timestamp('2024-01-01'), pd.Timestamp('2026-10-22')
bok = ch[ch.series_type == '한국은행 기준금리'][['date', 'rate_pct']].rename(columns={'date': 'applied_change_date'})
daily = pd.merge_asof(pd.DataFrame({'date': pd.date_range(START, END)}), bok,
                      left_on='date', right_on='applied_change_date', direction='backward')
assert daily.rate_pct.notna().all()
r = daily.set_index('date').rate_pct
print('일수(양끝 포함):', len(daily))
print(daily.iloc[[0, -1]].to_string(index=False))

DECISION_20261022 = 3.00   # 2026-10-22 결정문: Base Rate unchanged at 3.00%
last = ch.iloc[-1]
print(f'표의 최신 변경행: {last.date.date()} {last.rate_pct:.2f}% / 10/22 결정 {DECISION_20261022:.2f}% → 이번 결정의 변화 {DECISION_20261022 - last.rate_pct:+.2f}%p')

## Step 4. 2024년 이후 전체 흐름 (차트 1)

In [ ]:
win = ch[ch.date.between(START, END)]
print(f'2024-01-01~2026-10-22 변경 {len(win)}번: 인하 {(win.diff_pp < 0).sum()}번, 인상 {(win.diff_pp > 0).sum()}번')
fig, ax = plt.subplots(figsize=(12, 4.6))
ax.step(daily.date, daily.rate_pct, where='post', color=C['main'], lw=2.4)
for d, x, p in zip(win.date, win.rate_pct, win.diff_pp):
    ax.scatter(d, x, color=C['sub'] if p < 0 else C['main'], zorder=3, s=28)
    ax.annotate(f'{d:%y.%m.%d}\n{x:.2f}%', (d, x), xytext=(0, 9 if p > 0 else -26), textcoords='offset points', ha='center', fontsize=8)
ax.axvspan(pd.Timestamp('2026-01-01'), END, color=C['gray'], alpha=0.08)
ax.set_ylim(2.0, 3.75); ax.set_ylabel('기준금리 (연 %, 세로축 2.0%부터)')
ax.set_title('2024년 초 3.50%에서 2.50%까지 내려간 뒤, 2026년 두 번 올라 3.00%에서 멈췄다\n'
             '한국은행 기준금리 일별 상태, 2024-01-01~2026-10-22 (2026년은 10/22까지)', loc='left')
fig.text(0.01, -0.03, '자료: 한국은행 기준금리 추이 변경일자 표(2026-10-23 조회), 2026-10-22 통화정책방향 결정문', fontsize=8, color=C['gray'])
plt.show()

## Step 5. 실제로 바뀐 6번의 폭 (차트 2)

변경행 6개는 회의 6번이 아닙니다. 그 사이 동결한 회의는 표에 나오지 않습니다.

In [ ]:
print(win[['date', 'rate_pct', 'diff_pp']].to_string(index=False))
assert len(win) == 6 and set(win.diff_pp.abs().round(2)) == {0.25}
fig, ax = plt.subplots(figsize=(11, 4.4))
ax.bar(range(len(win)), win.diff_pp, color=[C['sub'] if p < 0 else C['main'] for p in win.diff_pp], width=0.55)
for i, p in enumerate(win.diff_pp):
    ax.text(i, p + (0.02 if p > 0 else -0.02), f'{p:+.2f}%p', ha='center', va='bottom' if p > 0 else 'top')
ax.axhline(0, color='#333', lw=0.8)
ax.set_xticks(range(len(win))); ax.set_xticklabels([f'{d:%Y-%m-%d}\n→ {x:.2f}%' for d, x in zip(win.date, win.rate_pct)], fontsize=9)
ax.set_ylim(-0.4, 0.4); ax.set_ylabel('직전 수준 대비 변화 (%p)')
ax.set_title('2024년 이후 금리가 바뀐 날은 6번, 모두 0.25%p씩이었다: 내림 4번 뒤 올림 2번\n'
             '변경일자별 변화폭 (2026년은 10/22까지, 10/22 동결은 변경행 아님)', loc='left')
fig.text(0.01, -0.03, '자료: 한국은행 기준금리 추이 변경일자 표(2026-10-23 조회)', fontsize=8, color=C['gray'])
plt.show()

## Step 6. 기준일을 바꾸면 부호가 바뀝니다 (차트 3)

같은 2026-10-22의 3.00%라도, 무엇과 비교하느냐에 따라 답이 달라집니다.

In [ ]:
v24, v26, vend = r['2024-01-01'], r['2026-01-01'], r['2026-10-22']
d26, d24 = round(vend - v26, 2), round(vend - v24, 2)
print(f'2024-01-01 {v24:.2f}% / 2026-01-01 {v26:.2f}% / 2026-10-22 {vend:.2f}%')
print(f'올해 초 대비 {d26:+.2f}%p, 2024년 초 대비 {d24:+.2f}%p, 이번 결정 변화 {DECISION_20261022 - last.rate_pct:+.2f}%p')
assert (v24, v26, vend, d26, d24) == (3.5, 2.5, 3.0, 0.5, -0.5)
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(['2024-01-01', '2026-01-01', '2026-10-22\n(동결 결정일)'], [v24, v26, vend], color=[C['gray'], C['sub'], C['main']], width=0.55)
for i, v in enumerate([v24, v26, vend]):
    ax.text(i, v + 0.06, f'{v:.2f}%', ha='center', fontsize=12, fontweight='bold')
ax.annotate('', xy=(2, 4.0), xytext=(0, 4.0), arrowprops=dict(arrowstyle='->', color='#333'))
ax.text(1, 4.08, f'2024년 초 대비 {d24:+.2f}%p', ha='center')
ax.annotate('', xy=(2, 4.55), xytext=(1, 4.55), arrowprops=dict(arrowstyle='->', color=C['main']))
ax.text(1.5, 4.63, f'2026년 초 대비 {d26:+.2f}%p', ha='center', color=C['main'])
ax.set_ylim(0, 5.0); ax.set_ylabel('기준금리 (연 %, 0부터)')
ax.set_title(f'같은 3.00%라도 올해 초보다 {d26:.2f}%p 높고, 2024년 초보다 {abs(d24):.2f}%p 낮다\n기준일별 기준금리 (변경일자 기준 일별 상태)', loc='left')
fig.text(0.01, -0.03, '자료: 한국은행 기준금리 추이 변경일자 표(2026-10-23 조회), 2026-10-22 통화정책방향 결정문', fontsize=8, color=C['gray'])
plt.show()

## Step 7. 올해는 어느 금리로 며칠을 보냈나 (차트 4)

2026-01-01부터 2026-10-22까지 양끝을 모두 포함해 셉니다. 올해가 끝나지 않았으므로 연간 값이 아닙니다.

In [ ]:
y26 = daily[daily.date >= '2026-01-01']
days = y26.groupby('rate_pct').size()
print(days.to_string(), '\n합계:', days.sum())
assert dict(days) == {2.5: 196, 2.75: 42, 3.0: 57} and days.sum() == 295
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.barh([f'{k:.2f}%' for k in days.index], days.values, color=[C['sub'], C['gray'], C['main']], height=0.55)
for i, v in enumerate(days.values):
    ax.text(v + 3, i, f'{v}일 ({v / days.sum() * 100:.1f}%)', va='center')
ax.invert_yaxis(); ax.set_xlim(0, 300); ax.set_xlabel('일수 (일, 1/1과 10/22 양끝 포함)')
ax.set_title(f'2026년(10/22까지) {days.sum()}일 중 {days[2.5]}일은 2.50%, 3.00%였던 날은 {days[3.0]}일\n2026년 기준금리 수준별 적용 일수 (부분 연도)', loc='left')
fig.text(0.01, -0.07, '자료: 한국은행 기준금리 추이 변경일자 표(2026-10-23 조회)', fontsize=8, color=C['gray'])
plt.show()

## 해석할 때 주의할 점

- 기준금리는 은행이 고객에게 받는 대출금리가 아닙니다. 이 결과로 개인의 이자 부담이나 앞으로의 금리를 예측할 수 없습니다.
- 표본이 아니라 공식 변경 기록 전체라 표본오차 개념이 없습니다. 원인(왜 올리고 내렸나)은 이 데이터로 판단하지 않습니다.
- 2026년 값은 10월 22일까지만 셌습니다.

## 부록: 한국은행 페이지에서 직접 다시 받기

기본으로 꺼져 있습니다(`RUN_COLLECT = False`). 키가 필요 없지만 페이지 구조가 바뀌면 동작하지 않을 수 있습니다. 표 `<tbody>`만 읽고, 페이지 안 그래프 배열은 쓰지 않습니다(조회 당시 그래프 배열의 마지막 점(2026/10/23)은 조회일과 같은 날짜이며 변경일이 아닙니다).

In [ ]:
RUN_COLLECT = False
URL = 'https://www.bok.or.kr/portal/singl/baseRate/list.do?dataSeCd=01&menuNo=200643'
def collect_bok_table(url=URL):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    html = urllib.request.urlopen(req, timeout=30).read().decode('utf-8')
    body = re.findall(r'<tbody>(.*?)</tbody>', html, re.S)
    assert len(body) == 1
    out = []
    for i, tr in enumerate(re.findall(r'<tr>(.*?)</tr>', body[0], re.S), 1):
        tds = [re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', '', t)).strip() for t in re.findall(r'<td[^>]*>(.*?)</td>', tr, re.S)]
        out.append({'table_row_order': i, 'year_text': tds[0], 'month_day_text': tds[1], 'base_rate_text': tds[2]})
    return pd.DataFrame(out)
if RUN_COLLECT:
    fresh = collect_bok_table()
    print(len(fresh), '행'); print(fresh.head())